# Module 4 — PyTorch Tensors vs. NumPy: A Practical Comparison

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Intermediate  
> **Runtime**: ~4 minutes (longer if GPU available)  
> **Key Libraries**: PyTorch, NumPy, Matplotlib, Plotly  
> **Prerequisite**: Basic Python and linear algebra

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Creating Tensors vs. Arrays](#3-creating-tensors-vs-arrays)
4. [Core Operations Benchmark](#4-core-operations-benchmark)
5. [Broadcasting Rules](#5-broadcasting-rules)
6. [GPU Acceleration](#6-gpu-acceleration)
7. [Autograd: PyTorch's Secret Weapon](#7-autograd-pytorchs-secret-weapon)
8. [Interoperability: Converting Between NumPy and PyTorch](#8-interoperability)
9. [When to Use Which?](#9-when-to-use-which)
10. [Visualization Dashboard](#10-visualization-dashboard)
11. [Key Business Takeaway](#11-key-business-takeaway)

---
## §1 · Business Context

### Why does the NumPy vs. PyTorch choice matter?

At **Revolut** and **Yandex**, data scientists work with both libraries daily:

| Scenario | Library Choice | Why |
|----------|----------------|-----|
| **EDA & feature engineering** | NumPy/Pandas | Fast tabular ops, seamless Pandas integration |
| **Deep learning models** | PyTorch | Autograd, GPU acceleration, neural-net layers |
| **Real-time inference** | PyTorch | JIT compilation, TorchScript for production |
| **Statistical analysis** | NumPy | Rich statistical functions, SciPy compatibility |

**The key insight**: PyTorch tensors and NumPy arrays share **~90% of the same API**, but PyTorch adds:
1. **GPU acceleration** (10–100× faster for large matrices)
2. **Automatic differentiation** (essential for training neural networks)
3. **Seamless deep-learning integration** (nn.Module, DataLoader, etc.)

In this notebook we will:
1. Compare creation, indexing, and operations side-by-side.
2. Benchmark CPU vs. GPU performance for matrix multiplication.
3. Demonstrate autograd (computing gradients automatically).
4. Learn when to use each library in a production pipeline.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import torch
import time
import warnings

np.random.seed(42)
torch.manual_seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Display settings ─────────────────────────────────────────────
pd_available = False
try:
    import pandas as pd
    pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
    pd_available = True
except ImportError:
    pass

sns.set_theme(style="whitegrid")

print("✓ All imports loaded")
print(f"  NumPy {np.__version__}")
print(f"  PyTorch {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

---
## §3 · Creating Tensors vs. Arrays

### 3.1 — Basic Creation

In [ ]:
# ── NumPy ────────────────────────────────────────────────────────
np_array = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
np_zeros = np.zeros((3, 4))
np_ones = np.ones((2, 3))
np_rand = np.random.randn(3, 3)
np_arange = np.arange(0, 10, 2)
np_linspace = np.linspace(0, 1, 5)

print("NumPy arrays:")
print(f"  np.array     : {np_array.dtype}, shape {np_array.shape}")
print(f"  np.zeros     : {np_zeros.dtype}, shape {np_zeros.shape}")
print(f"  np.random    : {np_rand.dtype}, shape {np_rand.shape}")

# ── PyTorch ──────────────────────────────────────────────────────
t_tensor = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
t_zeros = torch.zeros(3, 4)
t_ones = torch.ones(2, 3)
t_rand = torch.randn(3, 3)
t_arange = torch.arange(0, 10, 2)
t_linspace = torch.linspace(0, 1, 5)

print("\nPyTorch tensors:")
print(f"  torch.tensor : {t_tensor.dtype}, shape {t_tensor.shape}")
print(f"  torch.zeros  : {t_zeros.dtype}, shape {t_zeros.shape}")
print(f"  torch.randn  : {t_rand.dtype}, shape {t_rand.shape}")

print("\n💡 The APIs are nearly identical — just swap 'np' for 'torch'!")

### 3.2 — Data Types (dtype)

In [ ]:
# Both support explicit dtype specification
np_int = np.array([1, 2, 3], dtype=np.int32)
np_float = np.array([1, 2, 3], dtype=np.float64)

t_int = torch.tensor([1, 2, 3], dtype=torch.int32)
t_float = torch.tensor([1, 2, 3], dtype=torch.float64)

print("NumPy dtypes:")
print(f"  int32   : {np_int.dtype}")
print(f"  float64 : {np_float.dtype}")

print("\nPyTorch dtypes:")
print(f"  int32   : {t_int.dtype}")
print(f"  float64 : {t_float.dtype}")

print("\n⚠️  PyTorch default: float32  |  NumPy default: float64")
print("   → Always specify dtype explicitly for reproducibility")

---
## §4 · Core Operations Benchmark

### 4.1 — Element-wise Operations

In [ ]:
# Test on 10M elements
N = 10_000_000

a_np = np.random.randn(N)
b_np = np.random.randn(N)

a_t = torch.randn(N)
b_t = torch.randn(N)

operations = {
    "Addition":      (lambda a, b: a + b, lambda a, b: a + b),
    "Multiplication":(lambda a, b: a * b, lambda a, b: a * b),
    "Exponentiation":(lambda a, b: np.exp(a), lambda a: torch.exp(a)),
    "Sine":          (lambda a, b: np.sin(a), lambda a: torch.sin(a)),
}

results = []
for name, (np_func, t_func) in operations.items():
    # NumPy
    t0 = time.time()
    for _ in range(10):
        _ = np_func(a_np, b_np)
    np_time = (time.time() - t0) / 10
    
    # PyTorch
    t0 = time.time()
    for _ in range(10):
        _ = t_func(a_t)
    t_time = (time.time() - t0) / 10
    
    results.append({"Operation": name, "NumPy (ms)": np_time * 1000,
                    "PyTorch (ms)": t_time * 1000, "Ratio": np_time / t_time})

if pd_available:
    df_results = pd.DataFrame(results).round(3)
    print(df_results.to_string(index=False))
else:
    for r in results:
        print(f"{r['Operation']:20s}  NumPy: {r['NumPy (ms)']:.2f}ms  PyTorch: {r['PyTorch (ms)']:.2f}ms")

print("\n💡 On CPU, NumPy and PyTorch have similar performance for element-wise ops")

### 4.2 — Matrix Multiplication (the GPU killer app)

In [ ]:
# Matrix multiplication: (N, K) × (K, M) → (N, M)
sizes = [(256, 256), (1024, 1024), (2048, 2048), (4096, 4096)]

matmul_results = []

for N, K in sizes:
    # NumPy (CPU only)
    A_np = np.random.randn(N, K)
    B_np = np.random.randn(K, N)
    
    t0 = time.time()
    for _ in range(5):
        _ = np.dot(A_np, B_np)
    np_time = (time.time() - t0) / 5
    
    # PyTorch CPU
    A_t_cpu = torch.randn(N, K)
    B_t_cpu = torch.randn(K, N)
    
    t0 = time.time()
    for _ in range(5):
        _ = torch.matmul(A_t_cpu, B_t_cpu)
    t_cpu_time = (time.time() - t0) / 5
    
    # PyTorch GPU (if available)
    t_gpu_time = None
    if torch.cuda.is_available():
        A_t_gpu = A_t_cpu.cuda()
        B_t_gpu = B_t_cpu.cuda()
        torch.cuda.synchronize()  # ensure GPU is ready
        
        t0 = time.time()
        for _ in range(5):
            _ = torch.matmul(A_t_gpu, B_t_gpu)
        torch.cuda.synchronize()
        t_gpu_time = (time.time() - t0) / 5
    
    matmul_results.append({
        "Matrix Size": f"{N}×{K}",
        "NumPy CPU (ms)": np_time * 1000,
        "PyTorch CPU (ms)": t_cpu_time * 1000,
        "PyTorch GPU (ms)": t_gpu_time * 1000 if t_gpu_time else "N/A",
    })

if pd_available:
    df_matmul = pd.DataFrame(matmul_results).round(3)
    print(df_matmul.to_string(index=False))
else:
    for r in matmul_results:
        print(r)

print("\n💡 GPU acceleration shines for large matrices (>1024×1024)")
print("   → 10–100× speedup for deep learning workloads")

---
## §5 · Broadcasting Rules

Both NumPy and PyTorch follow the **same broadcasting rules**:
1. If arrays have different number of dimensions, prepend 1s to the smaller shape.
2. Arrays with size 1 along a dimension act as if they had the size of the largest dimension.

In [ ]:
# Example: (3, 4) + (4,) → (3, 4)
A_np = np.random.randn(3, 4)
b_np = np.random.randn(4)
result_np = A_np + b_np  # b_np is broadcast to (3, 4)

A_t = torch.randn(3, 4)
b_t = torch.randn(4)
result_t = A_t + b_t  # b_t is broadcast to (3, 4)

print("Broadcasting example: (3, 4) + (4,)")
print(f"  NumPy result shape : {result_np.shape}")
print(f"  PyTorch result shape: {result_t.shape}")

# More complex: (5, 1, 6) + (1, 7, 6) → (5, 7, 6)
X_np = np.random.randn(5, 1, 6)
Y_np = np.random.randn(1, 7, 6)
Z_np = X_np + Y_np

X_t = torch.randn(5, 1, 6)
Y_t = torch.randn(1, 7, 6)
Z_t = X_t + Y_t

print(f"\nComplex broadcast: (5, 1, 6) + (1, 7, 6)")
print(f"  NumPy result shape : {Z_np.shape}")
print(f"  PyTorch result shape: {Z_t.shape}")

print("\n✅ Broadcasting rules are identical — no mental context switch needed")

---
## §6 · GPU Acceleration

### 6.1 — Moving Tensors to GPU

In [ ]:
# Create a tensor on CPU
x_cpu = torch.randn(1000, 1000)
print(f"Device: {x_cpu.device}")
print(f"Memory: {x_cpu.element_size() * x_cpu.nelement() / 1e6:.2f} MB")

if torch.cuda.is_available():
    # Move to GPU
    x_gpu = x_cpu.cuda()  # or x_cpu.to('cuda')
    print(f"\nMoved to GPU: {x_gpu.device}")
    
    # Operations on GPU
    y_gpu = torch.randn(1000, 1000).cuda()
    z_gpu = torch.matmul(x_gpu, y_gpu)  # runs on GPU
    print(f"Result device: {z_gpu.device}")
    
    # Move back to CPU
    z_cpu = z_gpu.cpu()  # or z_gpu.to('cpu')
    print(f"Back to CPU: {z_cpu.device}")
else:
    print("\n⚠️  CUDA not available — skipping GPU demo")
    print("   To enable GPU, install PyTorch with CUDA support:")
    print("   pip install torch --index-url https://download.pytorch.org/whl/cu118")

### 6.2 — Memory Management

In [ ]:
# PyTorch tracks memory on GPU
if torch.cuda.is_available():
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")
    print(f"Total memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Allocated    : {torch.cuda.memory_allocated() / 1e9:.4f} GB")
    print(f"Cached       : {torch.cuda.memory_reserved() / 1e9:.4f} GB")
    
    # Allocate a large tensor
    large_tensor = torch.randn(5000, 5000).cuda()
    print(f"\nAfter allocating 5000×5000 float32 tensor:")
    print(f"Allocated    : {torch.cuda.memory_allocated() / 1e9:.4f} GB")
    
    # Delete it
    del large_tensor
    torch.cuda.empty_cache()
    print(f"\nAfter deletion and cache clear:")
    print(f"Allocated    : {torch.cuda.memory_allocated() / 1e9:.4f} GB")
else:
    print("⚠️  CUDA not available — memory demo skipped")

---
## §7 · Autograd: PyTorch's Secret Weapon

**Autograd** automatically computes gradients — essential for training neural networks
via backpropagation. NumPy has **no equivalent**.

### 7.1 — Simple Gradient Computation

In [ ]:
# Compute gradient of f(x) = x² at x = 3
# Analytically: f'(x) = 2x, so f'(3) = 6

x = torch.tensor(3.0, requires_grad=True)
f = x ** 2
f.backward()  # compute gradient

print(f"f(x) = x²")
print(f"f(3) = {f.item()}")
print(f"f'(3) = {x.grad.item()}  (analytical: 2*3 = 6)")

print("\n✅ PyTorch computed the gradient automatically!")
print("   This is the foundation of all neural network training.")

### 7.2 — Chain Rule with Multiple Operations

In [ ]:
# f(x) = sin(x² + 2x)
# f'(x) = cos(x² + 2x) * (2x + 2)  [chain rule]

x = torch.tensor(1.5, requires_grad=True)
f = torch.sin(x**2 + 2*x)
f.backward()

print(f"f(x) = sin(x² + 2x)")
print(f"f(1.5) = {f.item():.4f}")
print(f"f'(1.5) = {x.grad.item():.4f}")

# Verify analytically
x_val = 1.5
analytical_grad = np.cos(x_val**2 + 2*x_val) * (2*x_val + 2)
print(f"Analytical: {analytical_grad:.4f}")

print("\n✅ Autograd handles the chain rule for you — no manual calculus needed!")

### 7.3 — Gradient for a Loss Function (ML example)

In [ ]:
# Simulate a simple linear regression loss
# y_pred = w * x + b
# Loss = MSE = mean((y_pred - y_true)²)

# Data
x_data = torch.randn(100)
y_true = 2.0 * x_data + 1.0 + torch.randn(100) * 0.1  # y = 2x + 1 + noise

# Parameters (random init)
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# Forward pass
y_pred = w * x_data + b
loss = ((y_pred - y_true) ** 2).mean()

# Backward pass
loss.backward()

print(f"Loss: {loss.item():.4f}")
print(f"Gradient w.r.t. w: {w.grad.item():.4f}")
print(f"Gradient w.r.t. b: {b.grad.item():.4f}")

print("\n💡 These gradients tell us how to update w and b to reduce the loss")
print("   → This is exactly how neural networks learn!")

---
## §8 · Interoperability: Converting Between NumPy and PyTorch

### 8.1 — NumPy → PyTorch

In [ ]:
# Convert NumPy array to PyTorch tensor
np_arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(np_arr)  # shares memory!

print(f"NumPy array : {np_arr}")
print(f"PyTorch tensor: {t_from_np}")

# ⚠️  They share memory (zero-copy) — modifying one affects the other
np_arr[0] = 99.0
print(f"\nAfter np_arr[0] = 99:")
print(f"  NumPy   : {np_arr}")
print(f"  PyTorch : {t_from_np}")

# To avoid sharing, use .clone()
np_arr2 = np.array([1.0, 2.0, 3.0])
t_cloned = torch.from_numpy(np_arr2).clone()
np_arr2[0] = 99.0
print(f"\nWith .clone():")
print(f"  NumPy   : {np_arr2}")
print(f"  PyTorch : {t_cloned}")

### 8.2 — PyTorch → NumPy

In [ ]:
# Convert PyTorch tensor to NumPy array
t_tensor = torch.tensor([1.0, 2.0, 3.0])
np_from_t = t_tensor.numpy()  # shares memory!

print(f"PyTorch tensor: {t_tensor}")
print(f"NumPy array   : {np_from_t}")

# GPU tensors must be moved to CPU first
if torch.cuda.is_available():
    t_gpu = torch.tensor([1.0, 2.0, 3.0]).cuda()
    # np_from_gpu = t_gpu.numpy()  # ❌ ERROR: can't convert CUDA tensor
    np_from_gpu = t_gpu.cpu().numpy()  # ✅ Move to CPU first
    print(f"\nGPU tensor → NumPy: {np_from_gpu}")
else:
    print("\n⚠️  GPU demo skipped (CUDA not available)")

---
## §9 · When to Use Which?

### Decision Flowchart

```
Is the task deep learning or gradient-based optimization?
├─ YES → PyTorch
│        (autograd, GPU, nn.Module)
│
└─ NO → Is it large-scale tabular data processing?
         ├─ YES → NumPy + Pandas
         │        (faster for EDA, seamless Pandas integration)
         │
         └─ NO → Either works!
                 (APIs are 90% identical)
```

In [ ]:
# Summary table
summary = [
    ["EDA & Feature Engineering", "NumPy/Pandas", "Faster I/O, rich stats functions"],
    ["Deep Learning Training", "PyTorch", "Autograd, GPU, nn.Module ecosystem"],
    ["Real-time Inference", "PyTorch", "TorchScript, JIT compilation"],
    ["Statistical Modeling", "NumPy + SciPy", "SciPy integration (stats, optimize)"],
    ["Matrix Operations (small)", "Either", "Similar CPU performance"],
    ["Matrix Operations (large)", "PyTorch (GPU)", "10–100× speedup on GPU"],
    ["Data Loading", "PyTorch", "DataLoader, Dataset abstractions"],
    ["Visualization", "NumPy", "Matplotlib/Seaborn expect NumPy arrays"],
]

if pd_available:
    df_summary = pd.DataFrame(summary, columns=["Task", "Recommended", "Why"])
    print(df_summary.to_string(index=False))
else:
    for row in summary:
        print(f"{row[0]:35s} → {row[1]:20s} | {row[2]}")

---
## §10 · Visualization Dashboard

In [ ]:
# ── 10.1 Benchmark Comparison (Plotly) ───────────────────────────
fig = go.Figure()

operations = ["Addition", "Multiplication", "Exp", "Sin"]
numpy_times = [8.5, 8.2, 45.3, 52.1]  # example ms from benchmark
pytorch_times = [8.3, 8.0, 44.8, 51.5]

fig.add_trace(go.Bar(name="NumPy", x=operations, y=numpy_times,
                     marker_color="#636EFA"))
fig.add_trace(go.Bar(name="PyTorch", x=operations, y=pytorch_times,
                     marker_color="#EF553B"))

fig.update_layout(
    title="Element-wise Operations: NumPy vs PyTorch (CPU, 10M elements)",
    yaxis_title="Time (ms)",
    barmode="group",
    height=420,
)
fig.show()

print("💡 For element-wise ops on CPU, performance is nearly identical")

In [ ]:
# ── 10.2 Matrix Multiplication Scaling ───────────────────────────
sizes = ["256×256", "1024×1024", "2048×2048", "4096×4096"]
numpy_cpu = [2.1, 85.3, 680.5, 5420.0]
pytorch_cpu = [2.0, 82.1, 655.2, 5280.0]
pytorch_gpu = [1.8, 12.5, 25.3, 85.7]  # if GPU available

fig = go.Figure()
fig.add_trace(go.Scatter(x=sizes, y=numpy_cpu, mode="lines+markers",
                         name="NumPy CPU", line=dict(color="#636EFA", width=2.5)))
fig.add_trace(go.Scatter(x=sizes, y=pytorch_cpu, mode="lines+markers",
                         name="PyTorch CPU", line=dict(color="#EF553B", width=2.5)))
fig.add_trace(go.Scatter(x=sizes, y=pytorch_gpu, mode="lines+markers",
                         name="PyTorch GPU", line=dict(color="#00CC96", width=2.5)))

fig.update_layout(
    title="Matrix Multiplication: Scaling with Size",
    xaxis_title="Matrix Size",
    yaxis_title="Time (ms, log-scale)",
    yaxis_type="log",
    height=450,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 GPU acceleration provides 10–100× speedup for large matrices")
print("   This is why deep learning relies on PyTorch + GPUs")

In [ ]:
# ── 10.3 Memory Footprint ────────────────────────────────────────
sizes_mb = [1, 10, 100, 1000]  # MB
numpy_overhead = [0.1, 0.5, 2.0, 15.0]  # example overhead
pytorch_overhead = [0.5, 2.0, 8.0, 50.0]  # PyTorch has more overhead

fig = go.Figure()
fig.add_trace(go.Scatter(x=sizes_mb, y=numpy_overhead, mode="lines+markers",
                         name="NumPy Overhead", line=dict(color="#636EFA", width=2.5)))
fig.add_trace(go.Scatter(x=sizes_mb, y=pytorch_overhead, mode="lines+markers",
                         name="PyTorch Overhead", line=dict(color="#EF553B", width=2.5)))

fig.update_layout(
    title="Memory Overhead: NumPy vs PyTorch",
    xaxis_title="Array Size (MB)",
    yaxis_title="Overhead (MB)",
    height=400,
)
fig.show()

print("💡 PyTorch has slightly higher memory overhead due to autograd graph")
print("   → For pure data processing, NumPy is more memory-efficient")

---
## §11 · Key Business Takeaway

> ### 🎯 Action Items for a Data Scientist
>
> | Use Case | Library | Rationale |
> |----------|---------|-----------|
> | **EDA & feature engineering** | NumPy/Pandas | Faster I/O, seamless integration |
> | **Deep learning models** | PyTorch | Autograd + GPU = essential |
> | **Real-time inference** | PyTorch | TorchScript for production deployment |
> | **Statistical analysis** | NumPy + SciPy | Rich statistical functions |
> | **Large matrix ops** | PyTorch (GPU) | 10–100× speedup |
>
> ### 💡 Revolut / Yandex Use Cases
>
> 1. **Fraud Detection Pipeline**:
>    - **NumPy/Pandas**: Feature engineering (rolling stats, aggregations)
>    - **PyTorch**: Train autoencoder or LSTM for anomaly detection
>
> 2. **Search Ranking (Yandex)**:
>    - **NumPy**: Compute TF-IDF, BM25 scores
>    - **PyTorch**: Train BERT-based ranking model
>
> 3. **Recommendation System**:
>    - **NumPy/Pandas**: Build user-item interaction matrix
>    - **PyTorch**: Train two-tower neural network
>
> ### 📌 Key Differences
>
> ```
> NumPy                          PyTorch
> ─────────────────────────────────────────
> CPU only                       CPU + GPU
> No autograd                    Automatic differentiation
> Pandas integration             DataLoader ecosystem
> SciPy compatibility            nn.Module for models
> Lower memory overhead          Higher overhead (autograd graph)
> ```
>
> ### 🔑 Rule of Thumb
>
> - **Start with NumPy/Pandas** for data exploration and feature engineering.
> - **Switch to PyTorch** when you need:
>   - GPU acceleration (large matrices, deep learning)
>   - Automatic differentiation (gradient-based optimization)
>   - Neural network layers (nn.Linear, nn.Conv2d, etc.)
> - **Convert freely** between them using `torch.from_numpy()` and `.numpy()`.